# TI2Net: join the two extraction halves and check them

Inputs to attach (right panel -> Add Input): the outputs of **both** extraction versions, and `ti2net-assets`.
Accelerator: **None** (CPU is enough). Internet not needed.

Then **Save Version -> Save & Run All (Commit)**. Takes about 15-30 minutes.
Output: `FFpp_c23/real_ids.npy, real_lens.npy, fake_ids.npy, fake_lens.npy` (about 9.5 GB) + name lists + `check_report.txt`.


In [ ]:
import csv, glob, os, subprocess, sys
import numpy as np

ROOT = os.environ.get('TI2_INPUT', '/kaggle/input')
WORK = os.environ.get('TI2_WORK', '/kaggle/working')
HALVES = sorted(glob.glob(ROOT + '/**/id_vectors/FFpp_c23', recursive=True))
print('extraction outputs attached:')
for h in HALVES:
    print('  ', h)
assert len(HALVES) >= 2, 'attach BOTH extraction outputs (half 0/2 and half 1/2) as inputs'
auth = glob.glob(ROOT + '/**/authors_released/real_lens.npy', recursive=True)
AUTH = os.path.dirname(auth[0]) if auth else None
print('authors length files:', AUTH)
OUT = os.path.join(WORK, 'FFpp_c23')
SUBSETS = ['real', 'df', 'ff', 'fs', 'nt']
MIN_FRAMES = 131   # the official dataset code needs more than 130 usable frames per video
BAD = ['281', '344', '356', '370', '492', '509', '738', '908', '950']   # flagged by the authors
os.makedirs(os.path.join(WORK, 'src'), exist_ok=True)
os.makedirs(OUT, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/src/merge_ids.py
"""Merge per-video .npy files from extract_ids.py into the files TI2Net's train.py reads.

  python src/merge_ids.py --in_dirs half0/FFpp_c23 half1/FFpp_c23 --out_dir merged \
      --real real --fake df ff fs nt --min_len 0

--in_dirs: one or more extraction output folders (e.g. the two halves of a sharded run).
Writes real_ids.npy, real_lens.npy, fake_ids.npy, fake_lens.npy (+ *_names.txt) into --out_dir.
Videos are in sorted name order inside each subset; fake subsets are concatenated in the order given.
--min_len drops videos with fewer usable frames than this (0 keeps everything except empty videos).
The big array is written straight to disk, so memory use stays small.
"""
import argparse
import os
from glob import glob

import numpy as np


def collect(in_dirs, subset):
    """Sorted [(video_name, path)] for one subset across all input folders. A video found twice is an error."""
    found = {}
    for d in in_dirs:
        for f in glob(os.path.join(d, subset, "*.npy")):
            if f.endswith(".tmp.npy"):
                continue
            name = os.path.splitext(os.path.basename(f))[0]
            if name in found:
                raise SystemExit(f"{subset}/{name} is in two input folders: {found[name]} and {f}")
            found[name] = f
    return sorted(found.items())


def merge(in_dirs, out_dir, subsets, tag, min_len):
    keep, dropped = [], []
    for s in subsets:
        for name, f in collect(in_dirs, s):
            shape = np.load(f, mmap_mode="r").shape
            if shape[0] == 0 or shape[0] < min_len:
                dropped.append(f"{s}/{name}")
                continue
            keep.append((f"{s}/{name}", f, shape[0], shape[1]))
    assert keep, f"no videos found for {subsets} in {in_dirs}"
    dim = keep[0][3]
    assert all(k[3] == dim for k in keep), "vector sizes differ between videos"
    lens = np.array([k[2] for k in keep])
    os.makedirs(out_dir, exist_ok=True)
    out = np.lib.format.open_memmap(os.path.join(out_dir, f"{tag}_ids.npy"), mode="w+",
                                    dtype=np.float32, shape=(int(lens.sum()), dim))
    pos = 0
    for _, f, n, _ in keep:
        out[pos:pos + n] = np.load(f)
        pos += n
    out.flush()
    del out
    np.save(os.path.join(out_dir, f"{tag}_lens.npy"), lens)
    with open(os.path.join(out_dir, f"{tag}_names.txt"), "w") as fh:
        fh.write("\n".join(k[0] for k in keep))
    print(f"{tag}: {len(lens)} videos, {int(lens.sum())} frames, dim {dim}, dropped {len(dropped)} {dropped[:20]}")


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--in_dirs", nargs="+", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--real", nargs="+", default=["real"])
    p.add_argument("--fake", nargs="+", default=["df", "ff", "fs", "nt"])
    p.add_argument("--min_len", type=int, default=0)
    a = p.parse_args()
    merge(a.in_dirs, a.out_dir, a.real, "real", a.min_len)
    merge(a.in_dirs, a.out_dir, a.fake, "fake", a.min_len)


In [ ]:
report = []
def say(*a):
    line = ' '.join(str(x) for x in a)
    print(line); report.append(line)

ok = True
ours = {}
for s in SUBSETS:
    files, logs = {}, {}
    for h in HALVES:
        for f in glob.glob(os.path.join(h, s, '*.npy')):
            if f.endswith('.tmp.npy'):
                continue
            name = os.path.splitext(os.path.basename(f))[0]
            assert name not in files, f'{s}/{name} is in two halves'
            files[name] = f
        lp = os.path.join(h, s, '_log.csv')
        if os.path.exists(lp):
            for r in csv.DictReader(open(lp)):
                logs[r['video']] = {k: int(r[k]) for k in ['frames', 'faces', 'no_face', 'multi_face', 'fallback']}
    names = sorted(files)
    lens = np.array([np.load(files[n], mmap_mode='r').shape[0] for n in names])
    dims = {int(np.load(files[n], mmap_mode='r').shape[1]) for n in names}
    ours[s] = (names, lens)
    targets = sorted({n.split('_')[0] for n in names})
    tot = {k: sum(logs[n][k] for n in names if n in logs) for k in ['frames', 'faces', 'no_face', 'multi_face', 'fallback']}
    mism = [n for n, l in zip(names, lens) if n in logs and logs[n]['faces'] != l]
    short = [(n, int(l)) for n, l in zip(names, lens) if l < MIN_FRAMES]
    say(f'[{s}] videos {len(names)} | vectors {int(lens.sum())} | dim {dims} | {tot}')
    say(f'[{s}] shortest {int(lens.min())}, longest {int(lens.max())}, videos under {MIN_FRAMES} frames: {len(short)} {short[:30]}')
    worst = sorted(((logs[n]['no_face'], n) for n in names if n in logs), reverse=True)[:8]
    say(f'[{s}] most frames without a face (count, video): {worst}')
    bad = [(n, int(l), logs.get(n, {}).get('no_face')) for n, l in zip(names, lens) if n.split('_')[0] in BAD]
    say(f'[{s}] videos the authors flag as bad (name, vectors, no_face): {bad}')
    checks = {'1000 videos': len(names) == 1000, 'targets are 000..999': targets == [f'{i:03d}' for i in range(1000)],
              'vector size 1024': dims == {1024}, 'log has every video': all(n in logs for n in names),
              'file length = log faces': not mism}
    for k, v in checks.items():
        if not v:
            ok = False
            say(f'[{s}] CHECK FAILED: {k} {mism[:10] if k.startswith("file") else ""}')
say('ALL STRUCTURE CHECKS PASSED' if ok else 'SOME CHECKS FAILED - do not train on this yet')


In [ ]:
# Comparison with the authors' released length files (they used raw FF++ and kept every detected face).
if AUTH:
    ar = np.load(os.path.join(AUTH, 'real_lens.npy')); af = np.load(os.path.join(AUTH, 'fake_lens.npy'))
    auth = {'real': ar, 'df': af[0:1000], 'ff': af[1000:2000], 'fs': af[2000:3000], 'nt': af[3000:4000]}
    for s in SUBSETS:
        names, lens = ours[s]
        a = auth[s]
        say(f'[{s}] total: ours {int(lens.sum())}, authors {int(a.sum())}, ours/authors {lens.sum() / a.sum():.4f} | '
            f'median: ours {int(np.median(lens))}, authors {int(np.median(a))} | min: ours {int(lens.min())}, authors {int(a.min())}')
        if len(lens) == len(a):
            d = lens - a   # only meaningful if the authors' order is the sorted order (true for real and df at the start)
            say(f'[{s}] same position: equal {int((d == 0).sum())}, within 5 frames {int((np.abs(d) <= 5).sum())}, '
                f'within 5% {int((np.abs(d) <= 0.05 * a).sum())} of {len(a)}')
            big = np.argsort(-np.abs(d))[:10]
            say(f'[{s}] largest gaps (video, ours, authors): {[(names[i], int(lens[i]), int(a[i])) for i in big]}')
            say(f'[{s}] same lengths ignoring order: within 5 frames {int((np.abs(np.sort(lens) - np.sort(a)) <= 5).sum())} of {len(a)}')
else:
    say('authors length files not attached: comparison skipped')


In [ ]:
subprocess.run([sys.executable, os.path.join(WORK, 'src', 'merge_ids.py'), '--in_dirs', *HALVES, '--out_dir', OUT,
                '--real', 'real', '--fake', 'df', 'ff', 'fs', 'nt', '--min_len', '0'], check=True)

# Spot check: 40 random videos read back from the merged files must equal the per-video files.
rng = np.random.default_rng(0)
for tag, subs in [('real', ['real']), ('fake', ['df', 'ff', 'fs', 'nt'])]:
    ids = np.load(os.path.join(OUT, f'{tag}_ids.npy'), mmap_mode='r')
    lens = np.load(os.path.join(OUT, f'{tag}_lens.npy'))
    names = open(os.path.join(OUT, f'{tag}_names.txt')).read().split('\n')
    assert len(names) == len(lens) and lens.sum() == len(ids)
    starts = np.concatenate([[0], np.cumsum(lens)[:-1]])
    for i in rng.choice(len(lens), 40, replace=False):
        s, n = names[i].split('/')
        f = [p for h in HALVES for p in [os.path.join(h, s, n + '.npy')] if os.path.exists(p)][0]
        assert np.array_equal(ids[starts[i]:starts[i] + lens[i]], np.load(f)), names[i]
    say(f'{tag}: merged {len(lens)} videos, {len(ids)} vectors, finite {bool(np.isfinite(ids[::997]).all())}, spot check passed')
    del ids
open(os.path.join(OUT, 'check_report.txt'), 'w').write('\n'.join(report))
for f in sorted(os.listdir(OUT)):
    print(f, f'{os.path.getsize(os.path.join(OUT, f)) / 1e9:.2f} GB')
